# Automatisation du projet

Les pipelines partent déjà sur un push Git. Airflow ajoute une planification pour réentraîner le modèle sur des données récentes.
Le DAG crée un cluster Dataproc, lance le job PySpark, supprime le cluster et soumet Cloud Build.
Deux fichiers couvrent `staging` et `master`. Le build existant enchaîne MLflow et le déploiement.
Airflow exécute un DAG en fin de période : la fenêtre Spark utilise `ds` et `next_ds`.
La planification retenue est hebdomadaire, le lundi à 05:00.


## Dérive de modèle

La dérive conceptuelle apparaît quand la définition de la cible change : les prédictions suivent encore l'ancien sens de cette cible. La dérive des données apparaît quand la distribution des entrées change. Un modèle de durée de trajet calé hors été se dégrade en août, parce que la circulation n'a plus la même loi.

Le correctif opérationnel est un réentraînement sur des données récentes. Pour la prédiction d'achat, une fenêtre d'une à deux semaines garde le cycle semaine et week-end. Un événement ponctuel, par exemple les fêtes, ne domine pas l'échantillon. D'autres cas d'usage pondèrent les observations par ancienneté pour garder une mémoire plus longue.


## Place du DAG

Les pipelines déjà en place enchaînent les tests, Kedro, MLflow et le déploiement : Cloud Run en pré-production, Kubernetes en production. Airflow prépare les CSV, puis soumet le build de `purchase_predict`. Le reste suit le `cloudbuild.yaml` de la branche.

Les événements bruts arrivent en continu dans BigQuery. Le script `create_events_data_file.py` les transforme et les écrit dans le bucket lu par le pipeline Kedro `loading`.

Séquence : création du cluster, job PySpark sur la fenêtre du cycle, suppression du cluster et, en parallèle, Cloud Build. La planification `0 5 * * 1` vise chaque lundi à 05:00. L'intervalle du lundi 4 janvier s'exécute le lundi 11 janvier. Les arguments `{{ ds }}` et `{{ next_ds }}` couvrent donc du 4 au 11. Airflow exécute le DAG en fin de période : la fenêtre Spark utilise `ds` et `next_ds`.


## Connexion Google Cloud

Le paquet `apache-airflow-providers-google==5.0.0` fournit les opérateurs Dataproc et Cloud Build. Redémarrer le webserver et le scheduler pour que le type de connexion Google Cloud apparaisse, puis renseigner la connexion `google_cloud_default`.

`PROJECT_ID` est une variable Airflow, lue dans `default_args` et héritée par les tâches. `MLFLOW_SERVER` porte l'adresse joignable du serveur, lue au moment du build. Une variable dont le nom contient KEY ou SECRET est chiffrée.

Le compte de service dédié à Airflow reçoit les rôles Administrateur Dataproc, Utilisateur du compte de service, Lecteur des objets Cloud Storage, Lecteur de dépôt source et Compte de service Cloud Build. Déposer sa clé JSON dans la connexion.

`BUCKET` (`<NOM_DU_BUCKET>`) et `PROJECT_ID` doivent désigner le projet qui héberge le script `gs://<NOM_DU_BUCKET>/scripts/create_events_data_file.py`. La région `us-central1` limite le coût. `idle_delete_ttl` à 3600 secondes détruit un cluster inactif au bout d'une heure. Sous quota serré, `n1-standard-1` ou `n1-standard-2` remplace `n1-standard-4` dans `CLUSTER_CONFIG`. Le fichier conserve `n1-standard-4`.


## DAG de pré-production

Le fichier `pipeline_ml_staging.py` est le DAG complet. Le nom du DAG vient du nom du fichier. Le cluster s'appelle `purchase-predict-cluster-staging-` suivi de `{{ ds_nodash }}`.

`trigger_rule=ALL_DONE` supprime le cluster même en cas d'erreur. `gcloud builds submit` porte sur une archive pour conserver les dossiers vides Kedro (`conf/local`). `SHORT_SHA` est produit dans la tâche, sur 7 caractères, comme le tag d'image. Le corps du build Airflow a un timeout de 1800 s. Si le build interne `purchase_predict` atteint la limite par défaut, reprendre la machine `N1_HIGHCPU_8` et un timeout d'au moins 1200 s.

`task_delete_dataproc` est placé avant `task_trigger_build` dans la liste : sans parallélisme, le cluster est supprimé d'abord. Vérifier en fin d'exécution que le cluster est supprimé.


## Essais

`airflow tasks test` exécute une tâche seule. La date `2021-01-01` fixe le suffixe du cluster pour cet essai. Créer la variable `MLFLOW_SERVER` avant le backfill : le DAG complet lance aussi Cloud Build.

Le backfill du 6 au 13 novembre 2019 déclenche l'exécution du lundi 11 novembre. `tasks clear` retire les succès déjà enregistrés sur cet intervalle. Le calcul Spark et les builds peuvent prendre une trentaine de minutes. Les CSV produits sont ceux lus ensuite par Kedro.


## DAG de production

`pipeline_ml_production.py` reprend le même enchaînement. Le cluster se nomme `purchase-predict-cluster-production-`. Le clone Git se place sur `master`, et la substitution `BRANCH_NAME` vaut `master`. Le déploiement Kubernetes reste celui du `cloudbuild.yaml` de cette branche.

Les deux DAG dupliquent le calcul Spark, afin de planifier chaque environnement à part. Contrôler la suppression des deux clusters : ce sont des ressources facturées. Déposer les fichiers dans le dossier de DAG du serveur, avec le provider Google déjà installé.
